In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import transbigdata as tbd
import warnings
warnings.filterwarnings('ignore')
import pybdshadow
from shapely.geometry import Polygon

buildings_gdf = gpd.read_file('test_buildings.geojson')
""" 
MAPBOX_ACCESS_TOKEN = "pk.eyJ1IjoibmkxbzEiLCJhIjoiY2t3ZDgzMmR5NDF4czJ1cm84Z3NqOGt3OSJ9.yOYP6pxDzXzhbHfyk3uORg"
bounds = [139.803137,35.690984,139.804437,35.692684]
buildings_gdf = pybdshadow.get_buildings_by_bounds(139.804337,35.692584,139.804437,35.692684,MAPBOX_ACCESS_TOKEN)


buildings_gdf['x'] = buildings_gdf.centroid.x
buildings_gdf['y'] = buildings_gdf.centroid.y
buildings_gdf = buildings_gdf[(buildings_gdf['x'] > bounds[0]) &
                      (buildings_gdf['x'] <  bounds[2]) &
                      (buildings_gdf['y'] >  bounds[1]) &
                      (buildings_gdf['y'] <  bounds[3])]
buildings_gdf.plot() """

' \nMAPBOX_ACCESS_TOKEN = "pk.eyJ1IjoibmkxbzEiLCJhIjoiY2t3ZDgzMmR5NDF4czJ1cm84Z3NqOGt3OSJ9.yOYP6pxDzXzhbHfyk3uORg"\nbounds = [139.803137,35.690984,139.804437,35.692684]\nbuildings_gdf = pybdshadow.get_buildings_by_bounds(139.804337,35.692584,139.804437,35.692684,MAPBOX_ACCESS_TOKEN)\n\n\nbuildings_gdf[\'x\'] = buildings_gdf.centroid.x\nbuildings_gdf[\'y\'] = buildings_gdf.centroid.y\nbuildings_gdf = buildings_gdf[(buildings_gdf[\'x\'] > bounds[0]) &\n                      (buildings_gdf[\'x\'] <  bounds[2]) &\n                      (buildings_gdf[\'y\'] >  bounds[1]) &\n                      (buildings_gdf[\'y\'] <  bounds[3])]\nbuildings_gdf.plot() '

In [2]:
precision = 3600
date = '2022-01-01'

In [3]:

wallsunshine,pv_potential_building,pv_potential_wall = pybdshadow.cal_sunshine_facade(buildings_gdf, date, precision)

def offset_wall(wall_poly):
    wall_coords = np.array(wall_poly.exterior.coords)
    wall_coords[:,0]+=wall_coords[:,2]*0.000000001
    wall_coords[:,1]+=wall_coords[:,2]*0.000000001
    return Polygon(wall_coords)
wallsunshine['geometry'] = wallsunshine['geometry'].apply(offset_wall)

wallsunshine['type'] = 'wall'


roofsunshine,pv_potential_roof = pybdshadow.cal_sunshine(buildings_gdf,
                                   day=date,
                                   roof=True,
                                   accuracy='vector',
                                   precision=precision,
                                   areaonly=False
                                   )
roofsunshine['type'] = 'roof'
roofsunshine['geometry'] = roofsunshine.apply(lambda x: pybdshadow.extrude_poly(x['geometry'],x['height']),axis=1)

final_shadows_sunshinetime = pd.concat([#floorsunshine,
                                        wallsunshine,
                                        roofsunshine],axis=0)


Mem. usage decreased to  0.00 Mb (43.4% reduction)
Mem. usage decreased to  0.03 Mb (44.6% reduction)
Mem. usage decreased to  0.07 Mb (25.0% reduction)


In [3]:
shadow = pybdshadow.cal_sunshine(buildings_gdf,
                                   day=date,
                                   roof=True,
                                   accuracy='vector',
                                   precision=precision,
                                   areaonly=False
                                   )
shadow

,height,building_id,geometry,type,date
0,9.0,571,"POLYGON ((139.80408 35.69163, 139.80408 35.691...",roof,2021-12-31 22:21:43.483709696
0,9.0,46,"POLYGON ((139.80326 35.69154, 139.80326 35.691...",roof,2021-12-31 22:21:43.483709696
1,9.0,189,"POLYGON ((139.80335 35.69159, 139.80335 35.691...",roof,2021-12-31 22:21:43.483709696
2,9.0,268,"POLYGON ((139.80400 35.69179, 139.80400 35.691...",roof,2021-12-31 22:21:43.483709696
3,9.0,273,"MULTIPOLYGON (((139.80377 35.69132, 139.80377 ...",roof,2021-12-31 22:21:43.483709696
...,...,...,...,...,...
57,0.0,996,"POLYGON ((139.80344 35.69201, 139.80344 35.692...",ground,2022-01-01 06:21:43.483709696
58,0.0,1005,"POLYGON ((139.80326 35.69156, 139.80326 35.691...",ground,2022-01-01 06:21:43.483709696
59,0.0,1012,"POLYGON ((139.80328 35.69116, 139.80328 35.691...",ground,2022-01-01 06:21:43.483709696
60,0.0,1041,"POLYGON ((139.80340 35.69159, 139.80340 35.691...",ground,2022-01-01 06:21:43.483709696


In [4]:
pv_potential_roof.groupby('building_id')['roof_pv_potential'].sum().iloc[:50]

building_id
46     8.548574e+04
64     6.514712e+02
67     9.409704e+03
87     3.080046e+04
92     2.546472e+04
107    2.700573e+04
109    9.960935e+03
125    3.439771e+04
156    4.993831e+04
158    2.251377e+04
179    3.605375e-03
182    2.105219e+04
189    9.628799e+04
209    7.141994e+04
242    1.363877e+05
268    1.742635e+05
273    1.499266e+05
280    3.260498e+04
306    1.250329e+05
315    2.894716e+04
321    1.326068e+05
339    1.772362e+05
380    9.148201e+04
399    9.042625e+04
459    4.204588e+05
493    1.024246e+06
498    2.493688e+05
512    1.657201e+05
525    3.593674e+05
532    1.993191e+05
538    6.948135e+05
566    6.692206e+05
571    1.194268e+05
578    4.144403e+05
600    7.454954e+05
609    3.111790e+05
610    2.282626e+05
645    1.147722e+05
669    2.688024e+05
701    4.709021e+05
707    3.482630e+05
708    8.755017e+04
784    1.400327e+05
802    3.380906e+05
803    1.207858e+05
807    2.718089e+05
823    3.536874e+04
837    1.324551e+05
850    2.265895e+05
895    4

In [8]:
pv_potential_building

,building_id_left,pv_potential
0,46,97631.662642
1,64,42.275709
2,67,3215.527716
3,87,3259.628473
4,92,7991.739749
...,...,...
57,996,702112.517998
58,1005,10051.328869
59,1012,77501.035779
60,1041,892310.286520


In [9]:
pv_potential_wall

,building_id_left,target_wall_id,date,pv_potential
0,46,2,2021-12-31 22:21:43,-0.680944
1,46,2,2021-12-31 23:21:43,-4.137849
2,46,2,2022-01-01 00:21:43,3203.959542
3,46,2,2022-01-01 01:21:43,15726.270583
4,46,2,2022-01-01 02:21:43,21436.211398
...,...,...,...,...
9404,934,4,2022-01-01 06:21:43,0.000000
9410,934,5,2022-01-01 06:21:43,0.000000
9472,930,0,2022-01-01 06:21:43,0.000000
9492,930,5,2022-01-01 06:21:43,0.000000


In [10]:
final_shadows_sunshinetime.to_file('final_shadows_sunshinetime.geojson', driver='GeoJSON')

In [11]:
vis = pybdshadow.show_sunshine(sunshine = final_shadows_sunshinetime,
                  zoom='auto',vis_height = 1000)
vis

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(config={'version': 'v1', 'config': {'visState': {'filters': [], 'layers': [{'id': 'lz48o4', 'type': '…